# Notebook 5 — Product Association & Co-Purchase Network

**Goal:** Which products are mentioned together in reviews? This reveals cross-selling opportunities and natural product pairings that the Roastery can leverage.

**Method:** Extract product co-mentions from reviews and build a network graph showing which items are naturally paired by customers.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import re
from collections import Counter, defaultdict
from itertools import combinations
from pathlib import Path
from IPython.display import display

np.random.seed(42)

ON_KAGGLE = Path('/kaggle/working').exists()
if ON_KAGGLE:
    DATA_DIR = Path('/kaggle/input/starbucks-roastery-abm')
    if not DATA_DIR.exists():
        DATA_DIR = Path('/kaggle/input/datasets/shiratoriseto/starbucks-roastery-abm')
else:
    DATA_DIR = Path('../data/raw')

df = pd.read_csv(DATA_DIR / 'reviews_manual.csv')
print(f'Reviews: {len(df)}')

## Section 1 — Product Co-occurrence Matrix

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# PRODUCT EXTRACTION & CO-OCCURRENCE
# ══════════════════════════════════════════════════════════════════════

PRODUCT_PATTERNS = {
    'Espresso Martini': r'espresso\s+martini',
    'Matcha Margarita': r'matcha\s+margarita',
    'Old Fashioned': r'old\s+fashioned',
    'Cocktail Flight': r'(cocktail|martini|espresso)\s+flight',
    'Cold Brew Spiced Rum': r'(cold\s+brew\s+)?spiced\s+rum',
    'Whiskey Cloud': r'whiskey\s+cloud',
    'Cold Brew': r'cold\s+brew(?!\s+spiced)(?!\s+malt)',
    'Tiramisu Latte': r'tiramisu\s+latte',
    'Ube Latte': r'ube.{0,15}latte|ube.{0,15}coconut',
    'Masala Chai': r'masala\s+chai|chai\s+latte',
    'Matcha Latte': r'matcha\s+latte',
    'Affogato': r'affogato',
    'Espresso/Coffee Flight': r'(coffee|espresso|origin)\s+flight',
    'Siphon': r'siphon',
    'Croissant/Cornetto': r'croissant|cornetto',
    'Pizza/Focaccia': r'pizza|focaccia|flatbread',
    'Tiramisu (dessert)': r'tiramisu(?!\s+latte|\s+martini)',
    'Brownie': r'brownie',
    'Olive Oil Cake': r'olive\s+oil\s+cake',
    'Sandwich': r'sandwich|prosciutto|caprese|porchetta',
    'Merchandise': r'merch|mug|cup|sweatshirt|ornament',
    'Coffee Beans': r'(coffee\s+)?beans|bag\s+of',
}

def extract_products(text):
    found = []
    for name, pattern in PRODUCT_PATTERNS.items():
        if re.search(pattern, text, re.IGNORECASE):
            found.append(name)
    return found

df['products'] = df['text'].apply(extract_products)
df['n_products'] = df['products'].apply(len)

# Co-occurrence matrix
product_names = sorted(set(p for prods in df['products'] for p in prods))
cooc = pd.DataFrame(0, index=product_names, columns=product_names)

for products in df['products']:
    for p1, p2 in combinations(products, 2):
        cooc.loc[p1, p2] += 1
        cooc.loc[p2, p1] += 1

# Top co-occurrences
pairs = []
for i, p1 in enumerate(product_names):
    for j, p2 in enumerate(product_names):
        if i < j and cooc.loc[p1, p2] >= 2:
            pairs.append({'product_1': p1, 'product_2': p2, 'co_mentions': cooc.loc[p1, p2]})

pairs_df = pd.DataFrame(pairs).sort_values('co_mentions', ascending=False)
print(f'Product pairs with 2+ co-mentions: {len(pairs_df)}')
print('\n=== Top 15 Product Pairs ===')
display(pairs_df.head(15))

## Section 2 — Network Visualization

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# PRODUCT ASSOCIATION NETWORK
# ══════════════════════════════════════════════════════════════════════

# Product categories for coloring
PRODUCT_CATEGORIES = {
    'Espresso Martini': 'cocktail', 'Matcha Margarita': 'cocktail',
    'Old Fashioned': 'cocktail', 'Cocktail Flight': 'cocktail',
    'Cold Brew Spiced Rum': 'cocktail', 'Whiskey Cloud': 'cocktail',
    'Cold Brew': 'coffee', 'Tiramisu Latte': 'coffee',
    'Ube Latte': 'coffee', 'Masala Chai': 'coffee',
    'Matcha Latte': 'coffee', 'Affogato': 'coffee',
    'Espresso/Coffee Flight': 'coffee', 'Siphon': 'coffee',
    'Croissant/Cornetto': 'food', 'Pizza/Focaccia': 'food',
    'Tiramisu (dessert)': 'food', 'Brownie': 'food',
    'Olive Oil Cake': 'food', 'Sandwich': 'food',
    'Merchandise': 'retail', 'Coffee Beans': 'retail',
}

CAT_COLORS = {'cocktail': '#CBA258', 'coffee': '#00704A', 'food': '#E76F51', 'retail': '#457B9D'}

# Build network using spring layout
from collections import defaultdict
import math

# Only include products with at least 3 total mentions
product_freq = Counter(p for prods in df['products'] for p in prods)
active_products = [p for p, c in product_freq.items() if c >= 3]

# Simple force-directed layout (no networkx needed)
np.random.seed(42)
positions = {p: np.random.randn(2) for p in active_products}

# Spring layout iterations
for _ in range(100):
    forces = {p: np.zeros(2) for p in active_products}
    for p1 in active_products:
        for p2 in active_products:
            if p1 != p2:
                diff = positions[p1] - positions[p2]
                dist = max(np.linalg.norm(diff), 0.1)
                # Repulsion
                forces[p1] += diff / dist**2 * 0.5
                # Attraction (if co-occurring)
                weight = cooc.loc[p1, p2] if p1 in cooc.index and p2 in cooc.columns else 0
                if weight > 0:
                    forces[p1] -= diff * weight * 0.05
    for p in active_products:
        positions[p] += forces[p] * 0.1

# Plot
fig, ax = plt.subplots(figsize=(14, 10))

# Draw edges
for _, row in pairs_df.iterrows():
    p1, p2 = row['product_1'], row['product_2']
    if p1 in positions and p2 in positions:
        weight = row['co_mentions']
        ax.plot([positions[p1][0], positions[p2][0]], 
                [positions[p1][1], positions[p2][1]],
                'gray', alpha=min(weight/5, 0.8), linewidth=weight*0.8)

# Draw nodes
for p in active_products:
    cat = PRODUCT_CATEGORIES.get(p, 'other')
    color = CAT_COLORS.get(cat, '#999')
    size = product_freq[p] * 15
    ax.scatter(positions[p][0], positions[p][1], s=size, c=color, edgecolor='white', linewidth=1.5, zorder=5)
    ax.annotate(p, positions[p], fontsize=8, ha='center', va='bottom',
                xytext=(0, 8), textcoords='offset points')

# Legend
for cat, color in CAT_COLORS.items():
    ax.scatter([], [], c=color, s=100, label=cat.capitalize())
ax.legend(loc='upper left', fontsize=10)
ax.set_title('Product Co-mention Network (edge thickness = co-occurrence count)', fontsize=14, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.show()

## Section 3 — Cross-Floor Product Pairings

Which products from different floors are ordered together? These are cross-selling opportunities.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CROSS-FLOOR PRODUCT PAIRINGS
# ══════════════════════════════════════════════════════════════════════

# Map products to their primary floor
PRODUCT_FLOOR = {
    'Espresso Martini': '4F', 'Matcha Margarita': '4F', 'Old Fashioned': '4F',
    'Cocktail Flight': '4F', 'Cold Brew Spiced Rum': '4F', 'Whiskey Cloud': '4F',
    'Cold Brew': '1F', 'Tiramisu Latte': '1F', 'Ube Latte': '1F',
    'Masala Chai': '1F', 'Matcha Latte': '1F',
    'Affogato': '3F', 'Espresso/Coffee Flight': '3F', 'Siphon': '3F',
    'Croissant/Cornetto': '2F', 'Pizza/Focaccia': '2F', 'Sandwich': '2F',
    'Tiramisu (dessert)': '2F', 'Brownie': '2F', 'Olive Oil Cake': '2F',
    'Merchandise': '1F', 'Coffee Beans': '1F',
}

cross_floor_pairs = []
for _, row in pairs_df.iterrows():
    f1 = PRODUCT_FLOOR.get(row['product_1'], '?')
    f2 = PRODUCT_FLOOR.get(row['product_2'], '?')
    if f1 != f2 and f1 != '?' and f2 != '?':
        cross_floor_pairs.append({
            'product_1': row['product_1'], 'floor_1': f1,
            'product_2': row['product_2'], 'floor_2': f2,
            'co_mentions': row['co_mentions'],
            'cross_sell_opportunity': f'{f1} → {f2}',
        })

cross_df = pd.DataFrame(cross_floor_pairs).sort_values('co_mentions', ascending=False)
print(f'Cross-floor product pairings: {len(cross_df)}')
print('\n=== Top Cross-Floor Pairings (Cross-Sell Opportunities) ===')
if len(cross_df) > 0:
    display(cross_df.head(10))
else:
    print('  No cross-floor pairings found with 2+ co-mentions')

# Cross-sell matrix: which floor pairs have the most cross-selling potential?
if len(cross_df) > 0:
    cross_sell_matrix = cross_df.groupby('cross_sell_opportunity')['co_mentions'].sum().sort_values(ascending=False)
    print('\n=== Cross-Sell Potential by Floor Pair ===')
    for pair, count in cross_sell_matrix.items():
        print(f'  {pair}: {count} co-mentions')

## Section 4 — Actionable Cross-Sell Recommendations

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CROSS-SELL RECOMMENDATIONS
# ══════════════════════════════════════════════════════════════════════

print('=' * 70)
print('  CROSS-SELL RECOMMENDATIONS')
print('  Based on product co-mention analysis of 70 Yelp reviews')
print('=' * 70)

recommendations = [
    {
        'trigger': 'Customer orders Espresso Martini (4F)',
        'suggest': 'Tiramisu dessert (2F) or Pizza/Focaccia (2F)',
        'logic': 'Strong co-mention pattern: cocktail + food pairing',
        'implementation': 'App notification when ordering at 4F bar: "Pair your martini with our Tiramisu — pick up on 2F"',
    },
    {
        'trigger': 'Customer browsing 1F merchandise',
        'suggest': 'Coffee flight experience (3F)',
        'logic': 'Merch buyers are brand-engaged, likely to enjoy premium experiences',
        'implementation': 'Receipt/bag insert: "Complete your Reserve experience on 3F — try an Origin Flight"',
    },
    {
        'trigger': 'Customer orders Cocktail Flight (4F)',
        'suggest': 'Whiskey Barrel-Aged beans to take home (1F)',
        'logic': 'Flight → retail conversion: taste it, then buy the beans',
        'implementation': 'Bartender verbal suggestion + discount code for 1F purchase',
    },
    {
        'trigger': 'Customer eating on 2F (lunch)',
        'suggest': 'Matcha Margarita or Espresso Martini (4F)',
        'logic': 'Food → cocktail natural progression, especially afternoon',
        'implementation': 'Table card: "After lunch, head to 4F for our signature cocktails"',
    },
    {
        'trigger': 'Customer on 3F coffee experience',
        'suggest': 'Cold Brew Spiced Rum (4F)',
        'logic': 'Coffee → coffee cocktail progression for engaged customers',
        'implementation': '3F barista: "If you loved that origin, try it in a cocktail upstairs"',
    },
]

for i, rec in enumerate(recommendations, 1):
    print(f'\n  #{i}')
    print(f'  Trigger:        {rec["trigger"]}')
    print(f'  Suggest:        {rec["suggest"]}')
    print(f'  Logic:          {rec["logic"]}')
    print(f'  Implementation: {rec["implementation"]}')

## Limitations

1. **Co-mention ≠ co-purchase** — Mentioning two products in a review doesn't mean they were bought together in one visit
2. **Small sample (n=70)** — Rare product pairs may not appear at all
3. **Yelp reviewer bias** — Tends to highlight memorable/unique items over regular orders
4. **No basket-level data** — True market basket analysis requires POS transaction data

## What real data would add
- Actual basket-level co-purchase rates
- Time between purchases (same visit vs. return visit)
- Price sensitivity of cross-sell suggestions
- A/B test results of recommendation implementations